# Model Evaluation and Calibration

This notebook evaluates logistic regression event predictions produced by the model-training notebook. The goal is not to prove profitability. The goal is to measure whether predicted reversion probabilities are useful, precise, and reasonably calibrated for an ML trade filter.

The model is evaluated on labeled candidate events. Features are known at signal time, while labels were created from the forward reversion-before-stop rule.

In [ ]:
from pathlib import Path

import pandas as pd

from src.metrics import evaluate_predictions
from src.plotting import (
    plot_confusion_matrix,
    plot_precision_by_probability_bucket,
    plot_probability_calibration_curve,
)

try:
    from src.database import (
        connect_database,
        initialize_database,
        store_model_calibration_curve,
        store_model_confusion_matrix,
        store_model_evaluation_summary,
        store_probability_bucket_summary,
    )
except ImportError:
    connect_database = None

processed_dir = Path("../data/processed")
figures_dir = Path("../figures")
database_path = Path("../data/project.db")
schema_path = Path("../sql/schema.sql")

processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

## Load prediction data

The prediction table should come from the model-training notebook and contain one row per evaluated event. Required columns are the true label and the predicted reversion probability.

In [ ]:
pred_path = processed_dir / "predicted_reversion_probabilities.csv"

if not pred_path.exists():
    raise FileNotFoundError("Run the logistic regression notebook first or provide data/processed/predicted_reversion_probabilities.csv")

predictions = pd.read_csv(pred_path)
predictions.head()

## Evaluate classification and probability quality

Accuracy, precision, recall, ROC-AUC, and Brier score measure different weaknesses. For this project, precision is especially important because the model acts as a trade filter. Calibration matters because downstream steps may use probability thresholds or position sizing rules.

In [ ]:
evaluation = evaluate_predictions(
    predictions,
    probability_col="predicted_reversion_probability",
    label_col="label",
    split_col="split",
    threshold=0.50,
    n_bins=5,
)

summary = evaluation["model_evaluation_summary"]
confusion = evaluation["confusion_matrix"]
buckets = evaluation["probability_bucket_summary"]
calibration = evaluation["calibration_curve"]
roc_curve = evaluation["roc_curve"]

summary

## Confusion matrix

The confusion matrix shows the count of correct and incorrect classifications at the chosen probability threshold. A high false-positive count is damaging for a trade filter because it means the model admits failed events as trades.

In [ ]:
confusion

## Probability buckets and calibration

Probability buckets compare the model's predicted probability to the realized success rate. If the 0.60 to 0.80 bucket only succeeds 0.45 of the time, the model is overconfident in that region.

In [ ]:
buckets

In [ ]:
calibration

## Save outputs

The outputs are written as CSV tables and figures. The notebook overwrites placeholder artifacts when real model predictions are available.

In [ ]:
summary.to_csv(processed_dir / "model_evaluation_summary.csv", index=False)
confusion.to_csv(processed_dir / "confusion_matrix.csv", index=False)
buckets.to_csv(processed_dir / "precision_by_probability_bucket.csv", index=False)
calibration.to_csv(processed_dir / "probability_calibration_curve.csv", index=False)
roc_curve.to_csv(processed_dir / "roc_curve.csv", index=False)

plot_probability_calibration_curve(calibration, figures_dir / "probability_calibration_curve.png", split="validation")
plot_precision_by_probability_bucket(buckets, figures_dir / "precision_by_probability_bucket.png", split="validation")
plot_confusion_matrix(confusion, figures_dir / "confusion_matrix.png", split="validation")

figures_dir / "probability_calibration_curve.png

## Store evaluation outputs

The SQL tables preserve the evaluation layer so the later ML-filtered backtest can reference the exact probability model diagnostics used during model selection.

In [ ]:
if connect_database is not None and schema_path.exists():
    initialize_database(database_path, schema_path)
    with connect_database(database_path) as conn:
        store_model_evaluation_summary(conn, summary)
        store_model_confusion_matrix(conn, confusion)
        store_probability_bucket_summary(conn, buckets)
        store_model_calibration_curve(conn, calibration)

summary